In [ ]:
import os
import random
import shutil

def sample_dataset_pairs(src_dir, dest_dir, num_pairs):
    # Define source paths for both images and labels
    src_images = os.path.join(src_dir, "images")
    src_labels = os.path.join(src_dir, "labels")
    
    # Define destination paths
    dst_images = os.path.join(dest_dir, "images")
    dst_labels = os.path.join(dest_dir, "labels")
    
    # Gather all available images
    all_images = [
        f for f in os.listdir(src_images) 
        if os.path.isfile(os.path.join(src_images, f))
    ]
    
    # Prevent errors if requested sample size is too large
    sample_size = min(num_pairs, len(all_images))
    
    # Select a random sample of image files
    random_images = random.sample(all_images, sample_size)
    
    # Ensure both destination directories exist
    os.makedirs(dst_images, exist_ok=True)
    os.makedirs(dst_labels, exist_ok=True)
    
    # Process and copy the pairs
    for img_name in random_images:
        # 1. Copy the image
        img_src_path = os.path.join(src_images, img_name)
        img_dst_path = os.path.join(dst_images, img_name)
        shutil.copy2(img_src_path, img_dst_path)
        
        # 2. Find and copy the matching label
        # This splits 'photo1.jpg' into ('photo1', '.jpg') to find 'photo1' in labels
        base_name, _ = os.path.splitext(img_name)
        
        # Look for any file in the label folder that starts with the same base name
        label_file = None
        for f in os.listdir(src_labels):
            if os.path.splitext(f)[0] == base_name:
                label_file = f
                break
        
        # Copy the label if it exists
        if label_file:
            lbl_src_path = os.path.join(src_labels, label_file)
            lbl_dst_path = os.path.join(dst_labels, label_file)
            shutil.copy2(lbl_src_path, lbl_dst_path)
            #print(f"Copied pair: {img_name} <--> {label_file}")
        else:
            print(f"Warning: Label missing for image {img_name}")

# Example Usage:
# Your original training folder path
train_folder = r"..\data\license_plate_detection\train"
# Where you want to save the test sample
test_folder = r"..\data\formatted\license_plate_detection\random_train" 

#sample_dataset_pairs(train_folder, test_folder, 500)


In [ ]:
import os
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image

# ---------------------------------------------------------
# 1. Custom Dataset (YOLO to PyTorch Converter)
# ---------------------------------------------------------
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Load image and convert to PyTorch tensor [C, H, W] in range [0, 1]
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        image_tensor = F.to_tensor(image)
        
        label_name = img_name.replace('.jpg', '.txt')
        label_path = os.path.join(self.label_dir, label_name)
        
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Convert YOLO [x_c, y_c, w, h] to PyTorch [x_min, y_min, x_max, y_max]
                    x_min = (x_center - width / 2) * w
                    y_min = (y_center - height / 2) * h
                    x_max = (x_center + width / 2) * w
                    y_max = (y_center + height / 2) * h
                    
                    boxes.append([x_min, y_min, x_max, y_max])
                    
                    # CRITICAL: PyTorch requires class 0 to be background. 
                    # Shift YOLO classes up by 1.
                    labels.append(int(class_id) + 1)
                    
        # Handle images with no objects
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }
            
        return image_tensor, target

# Required to handle variable number of bounding boxes per image in a batch
def collate_fn(batch):
    return tuple(zip(*batch))

# ---------------------------------------------------------
# 2. Model Setup
# ---------------------------------------------------------
def get_object_detection_model(num_classes):
    # Load a model pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    
    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Replace the pre-trained head with a new one (tailored to your number of classes)
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

In [ ]:
def main():
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Using device: {device}")

    # --- CONFIGURATION ---
    # Number of classes = your classes + 1 (for background)
    # E.g., if you have 'cat', 'dog', 'bird' (3 classes), set this to 4.
    NUM_CLASSES = 2 
    BATCH_SIZE = 4
    NUM_EPOCHS = 2
    LEARNING_RATE = .00001
    MODEL_SAVE_PATH = r"..\models\custom_faster_rcnn.pt"
    
    # --- DATA PREP ---
    img_directory = r"..\data\formatted\license_plate_detection\random_train\images"
    label_directory = r"..\data\formatted\license_plate_detection\random_train\labels"
    dataset = YOLODataset(img_dir=img_directory, label_dir=label_directory)
    data_loader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=0, 
        collate_fn=collate_fn # Crucial for object detection
    )

    # --- MODEL & OPTIMIZER ---
    model = get_object_detection_model(NUM_CLASSES)
    model.to(device)
    
    # Load previously trained weights if you want to continue training
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"Loading existing model from {MODEL_SAVE_PATH} to continue training...")
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    else:
        print("No previous weights found. Starting fresh from COCO base...")

    # Construct an optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=LEARNING_RATE)

    # --- TRAINING LOOP ---
    print("Starting training...")
    for epoch in range(NUM_EPOCHS):
        model.train() # Set model to training mode
        epoch_loss = 0
        
        for i, (images, targets) in enumerate(data_loader):
            # Move images and targets to GPU/CPU
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Forward pass (Faster R-CNN returns a dictionary of losses during training)
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            # Backward pass
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            epoch_loss += losses.item()
            
            if i % 10 == 0:
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Batch [{i}/{len(data_loader)}] | Loss: {losses.item():.4f}")
                
        print(f"--- Epoch {epoch+1} Completed | Average Loss: {epoch_loss/len(data_loader):.4f} ---")

        # Save the model checkpoint after every epoch
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"Model saved to {MODEL_SAVE_PATH}\n")

In [ ]:
main()